# Thai organization list

Builds `data/thai_organizations.csv`, the reference list that `scrapping.ipynb` uses to fill the `ORG`
column (the organization a scam post is about). Run it once, and again only when you want a fresh
list: it downloads from three public sources, so it is the slowest step and does not need to run
with every scrape.

| Source | What it gives |
|---|---|
| [DGA open data, "รายชื่อหน่วยงานของรัฐส่วนกลางระดับกรมหรือเทียบเท่า"](https://catalog-dga.data.go.th/dataset/di-open1-01) | 413 central agencies (departments, state enterprises, independent agencies) with Thai name, English name and Thai abbreviation. Only a few rows carry an English abbreviation. Each ministry is added too, as its own row. |
| [Thai Wikipedia, รายชื่อธนาคารในประเทศไทย](https://th.wikipedia.org/wiki/รายชื่อธนาคารในประเทศไทย) | Banks currently operating, with the SET ticker or the state-bank abbreviation (BBL, KTB, SCB, GSB ...). |
| [Wikidata](https://www.wikidata.org) + English Wikipedia | English acronyms. The Thai name is looked up on Wikidata, which links to the English article, whose first sentence usually gives the acronym, e.g. "The Royal Thai Police (RTP) ...". An acronym is kept only if the article is about Thailand and the acronym fits the English name, because the bracket sometimes holds the parent ministry instead. |

Output columns: `name_th`, `name_en`, `abbr_th`, `abbr_en` (several joined with `|`), `type` (the
DGA category, or ธนาคาร for banks) and `source`. Most organizations have no English abbreviation,
because many have no English Wikipedia article or the article gives none.


In [ ]:
import io
import re
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (research scraper; Callas project)"}


In [ ]:
DGA_CSV = ("https://catalog-dga.data.go.th/dataset/af5f5723-0a54-47be-900f-19420585cc56/resource/"
           "64052093-d954-44f4-9993-d660f9fa245a/download/.csv")
BANKS_URL = "https://th.wikipedia.org/wiki/รายชื่อธนาคารในประเทศไทย"
ORG_PATH = Path("data/thai_organizations.csv")

has_thai = lambda s: bool(re.search(r"[฀-๿]", s))


def download(url, timeout=30):
    r = requests.get(url, headers=HEADERS, timeout=timeout)
    r.raise_for_status()
    return r


def load_dga_orgs():
    """Agencies, plus each ministry (the list only has its permanent-secretary office)."""
    d = pd.read_csv(io.BytesIO(download(DGA_CSV, timeout=60).content), encoding="utf-8-sig")
    # OrgNameAbbr mixes Thai and English abbreviations, e.g. "BOI, สกท."
    abbrs = d["OrgNameAbbr"].fillna("").map(lambda s: [a.strip() for a in s.split(",") if a.strip()])
    agencies = pd.DataFrame({
        "name_th": d["OrgNameThai"].str.strip(),
        "name_en": d["OrgNameEnglish"].astype(str).str.strip(),
        "abbr_th": abbrs.map(lambda a: "|".join(x for x in a if has_thai(x))),
        "abbr_en": abbrs.map(lambda a: "|".join(x for x in a if not has_thai(x))),
        "type": d["MajorCategoryName_byLaw"],
        "source": "DGA open data",
    })
    offices = d[d["OrgNameThai"].str.startswith("สำนักงานปลัดกระทรวง")].drop_duplicates("MinistryNameThai")
    office_abbr = dict(zip(offices["MinistryNameThai"], offices["OrgNameAbbr"].astype(str).str.strip()))
    ministries = pd.DataFrame([
        {"name_th": m.strip(), "name_en": "", "abbr_en": "", "type": "กระทรวง", "source": "DGA open data (ministry)",
         # สป.กค. (office of the ministry) -> กค. (the ministry)
         "abbr_th": office_abbr[m][3:].lstrip() if office_abbr.get(m, "").startswith("สป.") else ""}
        for m in sorted(d["MinistryNameThai"].dropna().unique()) if m.startswith("กระทรวง")
    ])
    return pd.concat([agencies, ministries])


def load_bank_orgs():
    rows = []
    soup = BeautifulSoup(download(BANKS_URL).text, "html.parser")
    for table in soup.select("table.wikitable")[:5]:  # banks still operating
        for tr in table.find_all("tr")[1:]:
            c = [td.get_text(" ", strip=True) for td in tr.find_all(["td", "th"])]
            if len(c) >= 5 and c[1].startswith("ธนาคาร"):
                m = re.search(r"(?:SET\s*:|ตัวอักษรย่อ\s*:)\s*([A-Z]+)", c[3])  # "SET: BBL" / "ตัวอักษรย่อ: GSB"
                rows.append({"name_th": c[1], "name_en": "", "abbr_th": "", "abbr_en": m.group(1) if m else "",
                             "type": "ธนาคาร", "source": "Thai Wikipedia"})
    return pd.DataFrame(rows)


# --- English acronyms: Thai name -> Wikidata item -> English Wikipedia article -> "(RTP)" in its first sentence
WIKI_HEADERS = {"User-Agent": "CallasResearch/0.1 (https://github.com/ZenithGeeks/Callas; scam dataset)"}
ACRONYM_IN_INTRO = re.compile(r"\((?:abbreviated(?: as)?:?\s*|abbr\.?:?\s*)?([A-Z][A-Za-z]*[A-Z]|[A-Z](?:\.[A-Z])+\.?)\)"
                              r"|abbreviated(?: as)? ([A-Z]{2,})")


def wiki_get(url, **params):
    """Wikimedia APIs ask for a descriptive User-Agent and sometimes rate limit, so retry."""
    for attempt in range(5):
        r = requests.get(url, params={**params, "format": "json"}, headers=WIKI_HEADERS, timeout=60)
        if r.ok and r.headers.get("content-type", "").startswith("application/json"):
            return r.json()
        time.sleep(3 * (attempt + 1))
    r.raise_for_status()
    raise RuntimeError(f"no JSON from {url}")


def acronym_fits(acronym, name):
    """RTP fits "Royal Thai Police": same first letter, letters in order, last letter starts a word."""
    words = re.sub(r"\([^)]*\)", "", name).split()
    words = words[1:] if words and words[0].lower() == "the" else words
    letters, text = re.sub(r"[^A-Za-z]", "", acronym).upper(), " ".join(words).upper()
    if not words or not letters or letters[0] != text[0] or letters[-1] not in {w[0].upper() for w in words}:
        return False
    rest = iter(text)
    return all(ch in rest for ch in letters)


def load_wikipedia_acronyms(orgs):
    """English acronyms of the organizations whose English Wikipedia article gives one."""
    item_names = {}  # Wikidata item -> Thai names with that label
    for i in range(0, len(orgs), 100):
        values = " ".join(f'"{n}"@th' for n in orgs["name_th"].iloc[i:i + 100])
        r = requests.post("https://query.wikidata.org/sparql", headers={**WIKI_HEADERS, "Accept": "application/json"},
                          data={"query": f"SELECT ?item ?th WHERE {{ VALUES ?th {{ {values} }} ?item rdfs:label ?th }}"},
                          timeout=120)
        r.raise_for_status()
        for b in r.json()["results"]["bindings"]:
            item_names.setdefault(b["item"]["value"].rsplit("/", 1)[-1], []).append(b["th"]["value"])

    items, article = list(item_names), {}  # Wikidata item -> English article title
    for i in range(0, len(items), 50):
        entities = wiki_get("https://www.wikidata.org/w/api.php", action="wbgetentities", ids="|".join(items[i:i + 50]),
                            props="sitelinks", sitefilter="enwiki")["entities"]
        article |= {q: e["sitelinks"]["enwiki"]["title"] for q, e in entities.items() if "enwiki" in e.get("sitelinks", {})}

    titles, intro = sorted(set(article.values())), {}  # article title -> plain-text intro
    for i in range(0, len(titles), 20):
        q = wiki_get("https://en.wikipedia.org/w/api.php", action="query", prop="extracts", exintro=1, explaintext=1,
                     exlimit=20, redirects=1, titles="|".join(titles[i:i + 20]))["query"]
        renamed = {x["from"]: x["to"] for x in q.get("normalized", []) + q.get("redirects", [])}
        text = {p["title"]: p.get("extract", "") for p in q["pages"].values()}
        for t in titles[i:i + 20]:
            intro[t] = text.get(renamed.get(renamed.get(t, t), renamed.get(t, t)), "")

    name_en = dict(zip(orgs["name_th"], orgs["name_en"]))
    rows = []
    for item, title in article.items():
        m = ACRONYM_IN_INTRO.search(intro[title][:350])
        if not m or "Thai" not in intro[title][:600]:  # no acronym, or an article about another country
            continue
        acronym = m.group(1) or m.group(2)
        for th in item_names[item]:
            # the bracket sometimes holds the parent ministry (e.g. MoNRE), so the acronym must fit the name
            if any(acronym_fits(acronym, n) for n in (name_en.get(th, ""), title) if n):
                rows.append({"name_th": th, "abbr_en": acronym, "source": "English Wikipedia"})
    return pd.DataFrame(rows)


first_filled = lambda s: next((v for v in s if v), "")
join_unique = lambda s: "|".join(dict.fromkeys(a for v in s for a in v.split("|") if a))


def merge_orgs(frames):
    """One row per Thai name; abbreviations from every source are combined."""
    return (pd.concat(frames, ignore_index=True).fillna("")
            .groupby("name_th", sort=False, as_index=False)
            .agg(name_en=("name_en", first_filled), abbr_th=("abbr_th", join_unique),
                 abbr_en=("abbr_en", join_unique), type=("type", "first"), source=("source", "first")))


orgs = merge_orgs([load_dga_orgs(), load_bank_orgs()])  # state banks are in both sources
orgs = merge_orgs([orgs, load_wikipedia_acronyms(orgs)])
ORG_PATH.parent.mkdir(parents=True, exist_ok=True)
orgs.to_csv(ORG_PATH, index=False, encoding="utf-8-sig")
print(f"{len(orgs)} organizations -> {ORG_PATH}; {(orgs['abbr_en'] != '').sum()} with an English abbreviation")
orgs["type"].value_counts()